# Tutorial 2: Creating a polymer simulation
### Learning Objectives
* Packing a simulation box
* Using forcefields to optimize positions
* Saving through GMSO
### Key Points
* Repeatably place many chains into a simulation box
* Easily mix polymers with other molecules or systems
* Ability to convert to gmso, which handles writing to various engine inputs

## Setup
------

In [ ]:
# Import necessary libraries
import mbuild as mb
from mbuild.polymer import Polymer
from mbuild.coordinate_transform import y_axis_transform
from mbuild.simulation import energy_minimize

# Check mBuild version
print(f"mBuild version: {mb.__version__}")

### Filling a simulation box
-----
Many viable methods for filling a box.

* Grid pack</br>
* Packmol random packing</br>
* Mixtures</br>


In [ ]:
# Create a polymer
polymer = Polymer()
polymer.add_monomer( # PEG
    mb.load("C{head}CO{tail}", smiles=True), separation=0.12,
    head_orientation=(0,1,0),
    tail_orientation=(0,-1,0)
)
polymer.build(10)


# Create align chain along Ly dimension
y_axis_transform(
    polymer, new_origin = polymer.children[0].children[0].pos, 
    point_on_y_axis=polymer.children[-1].children[0].pos
)
bbox = polymer.get_boundingbox()
pad = 1.5
polymer_radius  = bbox.Lx * pad # pad by 1.5 for space between chains
polymer_length = bbox.Ly
n_rows = int(polymer_length/polymer_radius*pad) - 2

# Create a grid
lattice = mb.Lattice(
    lattice_spacing = [polymer_radius, polymer_length, polymer_radius],
    lattice_vectors=[[1,0,0], [0,1,0], [0,0,1]], # xyz grid
    lattice_points={"Polymer": [[0,0,0]]}
)

# Pack
polymer_box = lattice.populate(x=n_rows, y=1, z=n_rows, compound_dict={"Polymer":polymer})

# Visualize

print(f"Full Box dimensions are: {polymer_box.get_boundingbox()}")
print(f"Single chain dimensions are: {polymer.get_boundingbox()}")
print(f"Created {polymer_box.n_particles} particle system")
polymer_box.visualize()

## Randomly pack a box
--------

In [ ]:
# Create a polymer
polymer = Polymer() # PEG
polymer.add_monomer(
    mb.load("C{head}CO{tail}", smiles=True), separation=0.12,
)
polymer.build(10)
energy_minimize(polymer)
bbox = polymer.get_boundingbox()
# y_axis_transform(
#     polymer, new_origin = polymer.children[0].children[0].pos, 
#     point_on_y_axis=polymer.children[-1].children[0].pos
# )

# Pack at low density of  2 g/ml
box = mb.Box([6,6,6])
polymer_box = mb.fill_box( 
    polymer, box=box, density=200,
)

# Visualize
print(f"Full Box dimensions are: {polymer_box.box}")
print(f"Single chain dimensions are: {polymer.get_boundingbox()}")
print(f"Created {polymer_box.n_particles} particle system")
polymer_box.visualize()

## Pack a mixture
-----

In [ ]:
# Create a polymer
polymer = Polymer()
polymer.add_monomer( # PEG
    mb.load("C{head}CO{tail}", smiles=True), separation=0.12,
)
polymer.build(50)
energy_minimize(polymer)
bbox = polymer.get_boundingbox()

# Pack at low density of  2 g/ml
box = mb.Box([6,6,6])
polymer_density = 200
polymer_box = mb.fill_box( 
    polymer, box=box, density=polymer_density,
)

# pack solvent # TODO: Use TIP4P system
solvent = mb.load("O", smiles=True) 
n_solvent = int((1000-polymer_density) * box.Lx**3 * 6.02e-1 / solvent.mass)
print(f"ADDING {n_solvent} water molecules to bring to a density of 1000 kg/m^3 ")
mixed_box = mb.solvate(polymer_box, solvent, n_solvent=n_solvent, box=box, edge=0)

# Visualize
print(f"Created {mixed_box.n_particles} particle system")
print(f"Final density is {mixed_box.mass / box.Lx**3/6.02e-1}")
mixed_box.visualize()

### Exercise: 